In [ ]:

import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from splitflow.StochasticAlgorithms import *

# Final Fig. 4 classes (FigureGeneration/VAE_pipeline/generate_synthetic_data.py):
# 4 s trajectories (sliding-window training), every draw kept (no arena rejection) for these four classes.
# The Lévy class is the optimal Lévy walk of generate_synthetic_data.levy_walk (not in Code/).
sys.path.insert(0, str((Path.cwd() / '../../FigureGeneration/VAE_pipeline').resolve()))
from generate_synthetic_data import levy_walk


In [ ]:
dt = 0.01
N = 10000
T=4   # final Brownian class (generator 'brownian', generate_synthetic_data.brownian): 4 s, no arena rejection
sims=[]
vels = []
np.random.seed(1234)
#for i in range(N):
i=0
while i < N:
    df = brownian_motion_sim(dt, T,i)
    i+=1    
    sims.append(df)


In [11]:
len(pd.concat(sims).obj_id_unique.unique())

10000

In [31]:
bm_sims = pd.concat(sims)

In [33]:
bm_sims['dataset_name']='brownian'

# random turns

In [ ]:
dfs = []
np.random.seed(84)
i=0
N=10000
while i < N:
    # final random-turns class (generator 'random_turns', generate_synthetic_data.random_turns): 0.2 s straights, turns U(60, 120) deg over
    # 0.12 s in a random direction, speed N(0.4, 0.15) per trajectory, turn schedule at a random phase per trajectory;
    # 4 s, no arena rejection
    times, x, y, vx, vy, angles = generate_random_turn_trajectory(total_time=4.0100001, straight_duration=0.2, direction=None,
                                                                  meanspeed=0.4, random_phase=True)
    times = np.arange(len(x)) * 0.01   # 401 samples, 0-4 s
    obj_id = 'random_turns_' + str(i)
    i+=1
    df = pd.DataFrame({'x':x, 'y':y, 'heading':np.arctan2(vy,vx), 'xvel': vx, 'yvel': vy, 'ground speed': np.sqrt(vx **2 + vy **2), 'time stamp': times*1000, 'obj_id_unique':obj_id})
    dfs.append(df)
    #plot_trajectory_with_velocities(times, x, y, vx, vy, angles)
        

In [221]:
random_turns_df_variable1= pd.concat(dfs)
              

# Levy Flight

In [ ]:
dfs = []
rng = np.random.default_rng(83)
i=0
N=10000
while i < N:
    # final Lévy class (generator 'levy_flight', generate_synthetic_data.levy): optimal Lévy walk, power-law segment
    # lengths (mu 2, l_min 10 mm) at one speed per trajectory |N(0.4, 0.15)| m/s, turn between segments uniform in
    # (-180, 180] deg over 50 ms, velocity noise N(0, 0.02) m/s; 4 s, no walls, no arena rejection
    s = max(0.01, abs(rng.normal(0.4, 0.15)))
    times, x, y, vx, vy = levy_walk(rng, s, T=4.0, dt=0.01, mu=2.0, l_min=0.01, turn_s=0.05, vel_noise=0.02)
    obj_id = 'levysmooth_' + str(i)
    i+=1
    df = pd.DataFrame({'x':x, 'y':y, 'heading':np.arctan2(vy,vx), 'xvel': vx, 'yvel': vy, 'ground speed': np.sqrt(vx **2 + vy **2), 'time stamp': times*1000, 'obj_id_unique':obj_id})
    dfs.append(df)
    #plot_trajectory_with_velocities(times, x, y, vx, vy, angles)
        

In [161]:
levy1 = pd.concat(dfs)
              

In [166]:
levy1['dataset_name']= 'levysmooth'

# Run and tumble

In [ ]:
times = []
pos = []
vel = []
obj_id =[]
sims  = []
N=10000
i=0
np.random.seed(42)

while i < N:
    print(i)
    # final run & tumble class (generator 'run_and_tumble', generate_synthetic_data.run_and_tumble(
    # redraw=False)): speed |N(0.4, 0.15)| m/s per trajectory, 4 s, no arena rejection
    speed = abs(np.random.normal(.4,.15))
    time, positions, velocities = run_and_tumble(
        total_time=4.0,
        dt=0.01,
        run_time=.05,
        tumble_time=.5,
        speed=speed,
       # seed=i
    )
    # velocity = central difference of position; positions scaled so the trajectory's mean speed is the drawn speed
    velocities = np.gradient(positions, 0.01, axis=0)
    k = speed / np.hypot(velocities[:,0], velocities[:,1]).mean()
    positions, velocities = positions * k, velocities * k
    obj_id_unique = 'run_and_tumble_' + str(i)
    times.append(time)
    pos.append(positions)
    vel.append(velocities)
    obj_id.append(obj_id_unique)
    df= pd.DataFrame({'x':positions[:,0], 'y': positions[:,1],
                             'xvel': velocities[:,0], 'yvel':velocities[:,1],
                                 'obj_id_unique': obj_id_unique,
                                 'time stamp': time*1000, })
    i+=1    
    sims.append(df)
    
    if i % 300 == 0:
        plt.plot(positions[:,0],positions[:,1])
        plt.title('Run and Tumble')
        plt.xlabel('x')
        plt.ylabel('y')
        plt.xlim(-.55,0.55)
        plt.ylim(-0.3,0.3)
        plt.gca().set_aspect('equal')

In [115]:
run_and_tumbledf = pd.concat(sims)

In [117]:
run_and_tumbledf['dataset_name']='run and tumble'